In [1]:
import pandas as pd
import sqlite3

print("=" * 80)
print("CREATING SQLITE DATABASE")
print("=" * 80)

# Read the complete CSV data
print("\n[1] Loading data...")
df = pd.read_csv('data/cleaned/worldbank_humanitarian_health_complete.csv')
print(f"   Loaded {len(df)} rows")

# Create SQLite database connection
print("\n[2] Creating database...")
conn = sqlite3.connect('humanitarian_health.db')
cursor = conn.cursor()

# Create countries table
print("\n[3] Creating countries table...")
cursor.execute('''
    CREATE TABLE IF NOT EXISTS countries (
        country_id INTEGER PRIMARY KEY AUTOINCREMENT,
        country_code TEXT UNIQUE NOT NULL,
        country_name TEXT NOT NULL,
        region TEXT
    )
''')

# Get unique countries
countries = df[['country_code', 'country_name', 'region']].drop_duplicates()
for idx, row in countries.iterrows():
    cursor.execute(
        "INSERT OR IGNORE INTO countries (country_code, country_name, region) VALUES (?, ?, ?)",
        (row['country_code'], row['country_name'], row['region'])
    )

print(f"   Loaded {len(countries)} countries")

# Create health indicators table
print("\n[4] Creating health_indicators table...")
cursor.execute('''
    CREATE TABLE IF NOT EXISTS health_indicators (
        indicator_id INTEGER PRIMARY KEY AUTOINCREMENT,
        country_id INTEGER NOT NULL,
        year INTEGER NOT NULL,
        under_5_mortality_rate REAL,
        life_expectancy REAL,
        maternal_mortality_ratio REAL,
        tuberculosis_incidence REAL,
        dpt_immunization_pct REAL,
        improved_sanitation_access_pct REAL,
        health_exp_per_capita_usd REAL,
        FOREIGN KEY (country_id) REFERENCES countries(country_id),
        UNIQUE(country_id, year)
    )
''')

# Load health indicators
for idx, row in df.iterrows():
    cursor.execute(
        "SELECT country_id FROM countries WHERE country_code = ?",
        (row['country_code'],)
    )
    country_id = cursor.fetchone()[0]
    
    cursor.execute(
        "INSERT OR IGNORE INTO health_indicators (country_id, year, under_5_mortality_rate, life_expectancy, maternal_mortality_ratio, tuberculosis_incidence, dpt_immunization_pct, improved_sanitation_access_pct, health_exp_per_capita_usd) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
        (country_id, row['year'], row['under_5_mortality_rate'], row['life_expectancy'], row['maternal_mortality_ratio'], row['tuberculosis_incidence'], row['dpt_immunization_pct'], row['improved_sanitation_access_pct'], row['health_exp_per_capita_usd'])
    )

print(f"   Loaded {len(df)} health records")

# Commit and close
conn.commit()

# Verify data
print("\n[5] Verifying data...")
cursor.execute("SELECT COUNT(*) FROM countries")
country_count = cursor.fetchone()[0]
print(f"   Countries: {country_count}")

cursor.execute("SELECT COUNT(*) FROM health_indicators")
indicator_count = cursor.fetchone()[0]
print(f"   Indicators: {indicator_count}")

cursor.execute("SELECT MIN(year), MAX(year) FROM health_indicators")
min_year, max_year = cursor.fetchone()
print(f"   Years: {min_year} to {max_year}")

conn.close()

print("\n" + "=" * 80)
print("DATABASE CREATED SUCCESSFULLY")
print("=" * 80)
print("\nDatabase file: humanitarian_health.db")
print("Ready for SQL queries")

CREATING SQLITE DATABASE

[1] Loading data...
   Loaded 195 rows

[2] Creating database...

[3] Creating countries table...
   Loaded 15 countries

[4] Creating health_indicators table...
   Loaded 195 health records

[5] Verifying data...
   Countries: 15
   Indicators: 195
   Years: 2010 to 2022

DATABASE CREATED SUCCESSFULLY

Database file: humanitarian_health.db
Ready for SQL queries
